# Specificity Inflation from scRNA-seq Sampling Noise

This notebook demonstrates **mechanistically** why a specificity cap is necessary:
for small cell types with low total UMI, Poisson/NB sampling noise creates
spuriously extreme specificity scores that don't reflect true underlying expression.

## Approach

1. **Fit noise model** from real single-cell data (mean-variance relationship)
2. **Simulate specificity** using real atlas parameters (461 cell types, actual UMI depths)
3. **Generate figures** showing inflation in low-UMI types
4. **Validate** against empirical `frac_clipped` from Specificity_Cap_Analysis

## Reference

Svensson 2020 (Genome Biology) showed that for 10x Chromium UMI data, excess
zeros are well-explained by NB/Poisson alone — true ZINB zero-inflation is minimal.
The "dropout" is expected from low expression + Poisson sampling.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK_DIR = Path().absolute()
PROJ_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ("notebooks_rebuttal", "notebooks") else NOTEBOOK_DIR
sys.path.insert(0, str(PROJ_DIR / "src"))

from CellType_PSY import Anno, Neur_idx, NonNeur_idx

FIG_DIR = PROJ_DIR / "results" / "figures" / "specificity_cap"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Plotting style ---
font_path = '/usr/share/fonts/truetype/msttcorefonts/Arial.ttf'
if Path(font_path).exists():
    fm.fontManager.addfont(font_path)
    fm._load_fontmanager(try_read_cache=False)

mpl.rcParams['figure.facecolor'] = 'none'
mpl.rcParams['axes.facecolor'] = 'none'
mpl.rcParams['savefig.facecolor'] = 'none'
mpl.rcParams['font.size'] = 12
mpl.rcParams['font.family'] = 'Arial'

# --- Data paths (absolute, external to repo) ---
# NOTE: These are on /mnt/data0/ and ~/Work/, not in the repo.
CLUSTER_DIR = Path("/mnt/data0/HumanBrainCellType/cluster_GeneXCell")
EXP_MAT_PATH = Path("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/Human.CT.Exp.Entrez.csv")
SPEC_UNCLIP_PATH = Path("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/dat/ExpMats/"
                         "HumanCT.TPM.0.1.Filt.Spec.clip100.0.lowexp.cut1e4.csv")

print(f"Clusters: {CLUSTER_DIR}")
print(f"Expression matrix: {EXP_MAT_PATH}")
print(f"Unclipped specificity: {SPEC_UNCLIP_PATH}")
print(f"Figures: {FIG_DIR}")

## Step 1: Estimate Noise Model from Single-Cell Data

For representative clusters spanning the UMI range, we fit the mean-variance
relationship per gene across cells:

$$\text{Var} = \mu + \frac{\mu^2}{\theta}$$

where $\theta$ is the NB overdispersion parameter. When $\theta \to \infty$,
the NB reduces to Poisson ($\text{Var} = \mu$).

### Representative clusters:
- **Small non-neuronal**: Cluster 0 (Miscellaneous/B-cell, 105 cells, UMI ~2,259)
- **Medium glia**: Cluster 44 (Oligodendrocyte, 101,039 cells, UMI ~5,994)
- **Large neuronal**: Cluster 200 (DG-GRAN, 23,293 cells, UMI ~10,000+)

In [ ]:
representative_clusters = {
    0: "Miscellaneous (B-cell)",
    44: "Oligodendrocyte",
    200: "DG granule neuron",
}


def fit_mean_variance(cluster_id, max_cells=2000):
    """Compute per-gene mean and variance from raw counts for a cluster.

    Uses usecols for fast loading of large clusters (reads only first max_cells).
    Returns DataFrame with columns: mean, var, dispersion_ratio (var/mean).
    """
    filepath = CLUSTER_DIR / f"{cluster_id}.csv.gz"

    # Read only the first max_cells+1 columns (col 0 = index) for efficiency
    cols_to_use = list(range(max_cells + 1))
    try:
        df = pd.read_csv(filepath, usecols=cols_to_use, index_col=0)
    except (ValueError, IndexError):
        # Cluster has fewer than max_cells columns; read entire file
        df = pd.read_csv(filepath, index_col=0)

    gene_mean = df.mean(axis=1)
    gene_var = df.var(axis=1, ddof=1)

    result = pd.DataFrame({"mean": gene_mean, "var": gene_var})
    result = result[result["mean"] > 0].copy()
    result["dispersion_ratio"] = result["var"] / result["mean"]
    return result


def nb_variance(mu, theta):
    """NB variance: Var = mu + mu^2/theta."""
    return mu + mu**2 / theta


def fit_nb_theta(mv_df, mu_bins=30):
    """Fit global theta from mean-variance data using binned medians."""
    mv = mv_df[(mv_df["mean"] > 0.01) & (mv_df["var"] > 0)].copy()
    mv["log_mean"] = np.log10(mv["mean"])

    mv["bin"] = pd.cut(mv["log_mean"], bins=mu_bins)
    binned = mv.groupby("bin", observed=True).agg(
        mu=("mean", "median"),
        var_median=("var", "median"),
        count=("mean", "size"),
    ).dropna()
    binned = binned[binned["count"] >= 10]

    mu_vals = binned["mu"].values
    var_vals = binned["var_median"].values

    try:
        popt, _ = curve_fit(nb_variance, mu_vals, var_vals,
                            p0=[1.0], bounds=(0.01, 1000))
        theta = popt[0]
    except RuntimeError:
        theta = np.nan

    return theta, mu_vals, var_vals


print("Fitting noise models for representative clusters...")
noise_model_results = {}

for cid, cname in representative_clusters.items():
    print(f"\n  Cluster {cid} ({cname}):")
    mv = fit_mean_variance(cid)
    theta, mu_binned, var_binned = fit_nb_theta(mv)
    n_cells = Anno.loc[cid, "Number of cells"]
    total_umi = Anno.loc[cid, "Total UMI"]
    median_disp = mv["dispersion_ratio"].median()

    print(f"    N_cells={n_cells:.0f}, Total_UMI={total_umi:.0f}")
    print(f"    Median Var/Mean = {median_disp:.2f}")
    print(f"    Fitted theta = {theta:.3f}")

    noise_model_results[cid] = {
        "name": cname,
        "n_cells": n_cells,
        "total_umi": total_umi,
        "theta": theta,
        "median_disp_ratio": median_disp,
        "mv_df": mv,
        "mu_binned": mu_binned,
        "var_binned": var_binned,
    }

### Mean-Variance Relationship Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (cid, res) in enumerate(noise_model_results.items()):
    ax = axes[idx]
    mv = res["mv_df"]
    theta = res["theta"]

    n_plot = min(5000, len(mv))
    np.random.seed(42)
    plot_idx = np.random.choice(len(mv), n_plot, replace=False)
    ax.scatter(mv["mean"].values[plot_idx], mv["var"].values[plot_idx],
               alpha=0.1, s=5, color="steelblue", rasterized=True)

    mu_range = np.logspace(np.log10(mv["mean"].min()), np.log10(mv["mean"].max()), 100)
    ax.plot(mu_range, mu_range, "k--", lw=1.5, label="Poisson (Var = μ)")

    if not np.isnan(theta):
        ax.plot(mu_range, nb_variance(mu_range, theta), "r-", lw=2,
                label=f"NB (θ = {theta:.2f})")

    ax.scatter(res["mu_binned"], res["var_binned"],
               color="orange", s=40, zorder=5, edgecolors="black", lw=0.5,
               label="Binned medians")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Mean count (μ)")
    ax.set_ylabel("Variance")
    ax.set_title(f"Cluster {cid}: {res['name']}\n"
                 f"(UMI={res['total_umi']:.0f}, N={res['n_cells']:.0f})",
                 fontsize=11)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.8)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Mean-Variance Relationship in Single-Cell Data",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "zinb_simulation_mean_variance.pdf",
            dpi=300, transparent=True, bbox_inches="tight")
fig.savefig(FIG_DIR / "zinb_simulation_mean_variance.png",
            dpi=300, transparent=True, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR}")

In [ ]:
noise_params = pd.DataFrame([
    {
        "cluster_id": cid,
        "name": res["name"],
        "n_cells": res["n_cells"],
        "total_umi": res["total_umi"],
        "theta": res["theta"],
        "median_var_over_mean": res["median_disp_ratio"],
    }
    for cid, res in noise_model_results.items()
])
noise_params.to_csv(FIG_DIR / "noise_model_fit.csv", index=False)
print("Noise model parameters:")
print(noise_params.to_string(index=False))

### Interpretation

- **Var/Mean > 1** for all clusters confirms NB (not Poisson) is needed
- Low-UMI clusters show higher overdispersion, meaning sampling noise is worse
- The fitted θ values give us empirically grounded noise for simulation

## Step 2: Simulate Specificity with Real Atlas Parameters

For a gene with **uniform true expression** across all 461 cell types
(i.e., same fraction of total UMI in every cluster), we simulate:

1. For each cell type, draw N_cells individual cells (vectorized)
2. Each cell: total UMI ~ lognormal (matching cluster's UMI distribution)
3. Gene count per cell ~ NB(μ = true_fraction × total_UMI, θ)
4. Compute cluster-average count, then TPM, then specificity

Expected result: specificity should be ~1.0 for all cell types (uniform expression).
Inflation in low-UMI types = artifact from sampling noise.

In [ ]:
# Load cluster-average expression (ground truth)
exp_mat = pd.read_csv(EXP_MAT_PATH, index_col=0)
exp_mat.columns = [int(c) for c in exp_mat.columns]
print(f"Expression matrix: {exp_mat.shape[0]} genes × {exp_mat.shape[1]} cell types")

cluster_total_exp = exp_mat.sum(axis=0)
print(f"Cluster total expression range: {cluster_total_exp.min():.1f} - {cluster_total_exp.max():.1f}")

In [ ]:
global_theta = np.median([res["theta"] for res in noise_model_results.values()
                          if not np.isnan(res["theta"])])
print(f"Global theta for simulation: {global_theta:.3f}")


def simulate_uniform_gene_specificity(
    anno, exp_mat, n_sims=500, theta=None, seed=42,
    n_cells_override=None, min_tpm=0.1, max_cells_per_type=500,
):
    """Simulate specificity for uniformly-expressed genes using real atlas parameters.

    Vectorized: simulates all cell types at once using fixed n_cells per type.

    Parameters
    ----------
    anno : DataFrame
        Cell type annotations with 'Total UMI' and 'Number of cells'
    exp_mat : DataFrame
        Cluster-average expression matrix (genes × cell types)
    n_sims : int
        Number of simulated genes (each independently sampled)
    theta : float or None
        NB overdispersion. None = Poisson.
    seed : int
        Random seed
    n_cells_override : int or None
        If set, use this many cells per type instead of real counts
    min_tpm : float
        TPM cutoff (values below this are set to 0)
    max_cells_per_type : int
        Cap cells per type for compute efficiency

    Returns
    -------
    result_df : DataFrame with per-cell-type stats
    all_specs : ndarray (n_sims × n_cts)
    """
    rng = np.random.default_rng(seed)
    ct_ids = sorted(anno.index)
    n_cts = len(ct_ids)

    total_umis = np.array([anno.loc[ct, "Total UMI"] for ct in ct_ids], dtype=float)
    n_cells_real = np.array([anno.loc[ct, "Number of cells"] for ct in ct_ids], dtype=float)

    if n_cells_override is not None:
        nc = n_cells_override
    else:
        nc = max_cells_per_type

    # Gene expression fraction: uniform across cell types
    # Use median nonzero gene expression / mean cluster total
    cluster_totals = np.array([exp_mat[ct].sum() for ct in ct_ids])
    true_frac = np.median(exp_mat.values[exp_mat.values > 0]) / cluster_totals.mean()

    # Vectorized simulation with chunking for memory management
    # For nc > 1000, process in chunks to avoid OOM
    chunk_size = min(nc, 1000)
    n_chunks = (nc + chunk_size - 1) // chunk_size

    log_total_umis = np.log(total_umis)[np.newaxis, :, np.newaxis]  # (1, n_cts, 1)

    # Accumulate sums across chunks (mean = sum / nc)
    sum_gene_counts = np.zeros((n_sims, n_cts))
    sum_total_umis = np.zeros((n_sims, n_cts))
    cells_done = 0

    for chunk_i in range(n_chunks):
        chunk_nc = min(chunk_size, nc - cells_done)

        cell_total_umis = rng.lognormal(
            mean=log_total_umis, sigma=0.3,
            size=(n_sims, n_cts, chunk_nc),
        )
        mu_per_cell = true_frac * cell_total_umis

        if theta is not None and theta < 1000:
            p = theta / (theta + mu_per_cell)
            p = np.clip(p, 1e-10, 1 - 1e-10)
            gene_counts = rng.negative_binomial(n=theta, p=p).astype(float)
        else:
            gene_counts = rng.poisson(mu_per_cell).astype(float)

        sum_gene_counts += gene_counts.sum(axis=2)
        sum_total_umis += cell_total_umis.sum(axis=2)
        cells_done += chunk_nc

    cluster_avg_counts = sum_gene_counts / nc       # (n_sims, n_cts)
    cluster_avg_total_umi = sum_total_umis / nc     # (n_sims, n_cts)

    # TPM
    tpm = (cluster_avg_counts / (cluster_avg_total_umi + 1e-12)) * 1e6
    tpm[tpm < min_tpm] = 0

    # Specificity: S = (TPM / sum(TPM)) * N_cts
    total_tpm = tpm.sum(axis=1, keepdims=True)
    all_specs = np.where(total_tpm > 0, (tpm / total_tpm) * n_cts, 0)

    # Aggregate stats per cell type
    result_df = pd.DataFrame({
        "ct": ct_ids,
        "total_umi": total_umis,
        "n_cells": n_cells_real,
        "mean_spec": all_specs.mean(axis=0),
        "std_spec": all_specs.std(axis=0),
        "median_spec": np.median(all_specs, axis=0),
        "p95_spec": np.percentile(all_specs, 95, axis=0),
        "is_neuronal": [ct in Neur_idx for ct in ct_ids],
    })

    return result_df, all_specs

In [ ]:
print("Running main simulation (500 uniform genes, vectorized)...")
sim_result, all_specs = simulate_uniform_gene_specificity(
    Anno, exp_mat,
    n_sims=500,
    theta=global_theta,
    seed=42,
)

print(f"\nSimulation complete. {len(sim_result)} cell types.")
print(f"Mean specificity (should be ~1.0): {sim_result['mean_spec'].mean():.3f}")
print(f"Range: {sim_result['mean_spec'].min():.3f} - {sim_result['mean_spec'].max():.3f}")

top_inflated = sim_result.nlargest(10, "mean_spec")
print("\nTop 10 most inflated cell types:")
for _, row in top_inflated.iterrows():
    ct = int(row["ct"])
    sc = Anno.loc[ct, "Supercluster"]
    neur = "NEUR" if row["is_neuronal"] else "non-NEUR"
    print(f"  CT {ct}: mean_spec={row['mean_spec']:.3f}, UMI={row['total_umi']:.0f}, "
          f"N={row['n_cells']:.0f} [{sc}] ({neur})")

## Step 3: Publication-Ready Figures

- **Panel A**: Specificity vs Total UMI for uniform genes
- **Panel B**: Effect of N_cells (50, 100, 500, 5000) on specificity noise
- **Panel C**: Effect of cap levels (1×, 2×, 3×, uncapped)
- **Panel D**: Validation — simulated inflation vs empirical frac_clipped

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# =========================================================================
# Panel A: Specificity vs Total UMI
# =========================================================================
ax = axes[0, 0]
neur_mask_sim = sim_result["is_neuronal"]

ax.scatter(
    sim_result.loc[neur_mask_sim, "total_umi"],
    sim_result.loc[neur_mask_sim, "mean_spec"],
    color="red", alpha=0.4, s=25, edgecolors="white", lw=0.3,
    label="Neuronal", zorder=3,
)
ax.scatter(
    sim_result.loc[~neur_mask_sim, "total_umi"],
    sim_result.loc[~neur_mask_sim, "mean_spec"],
    color="blue", alpha=0.6, s=25, edgecolors="white", lw=0.3,
    label="Non-neuronal", zorder=4,
)

ax.axhline(y=1.0, color="gray", ls="--", lw=1.5, alpha=0.7, label="True specificity (1.0)")

# Label top 5 most inflated
for _, row in sim_result.nlargest(5, "mean_spec").iterrows():
    ct = int(row["ct"])
    sc = Anno.loc[ct, "Supercluster"]
    short_name = sc[:15] + "..." if len(sc) > 15 else sc
    ax.annotate(
        f"{short_name}\n(UMI={row['total_umi']:.0f})",
        xy=(row["total_umi"], row["mean_spec"]),
        fontsize=7, ha="left",
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.5),
        xytext=(10, 5), textcoords="offset points",
    )

# Label largest neuronal
largest_neur = sim_result[neur_mask_sim].nlargest(1, "total_umi").iloc[0]
ct_lg = int(largest_neur["ct"])
sc_lg = Anno.loc[ct_lg, "Supercluster"]
ax.annotate(
    f"{sc_lg[:15]}\n(UMI={largest_neur['total_umi']:.0f})",
    xy=(largest_neur["total_umi"], largest_neur["mean_spec"]),
    fontsize=7, ha="right",
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5),
    xytext=(-10, 5), textcoords="offset points",
)

ax.set_xscale("log")
ax.set_xlabel("Total UMI per cell type")
ax.set_ylabel("Simulated specificity\n(uniform gene, expected = 1.0)")
ax.set_title("A", fontweight="bold", loc="left", fontsize=16)
ax.legend(fontsize=9, framealpha=0.8, loc="upper right")

rho_sim, p_sim = stats.spearmanr(sim_result["total_umi"], sim_result["mean_spec"])
ax.text(0.03, 0.97, f"ρ = {rho_sim:.3f}\np = {p_sim:.1e}",
        transform=ax.transAxes, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# =========================================================================
# Panel B: Effect of N_cells
# =========================================================================
ax = axes[0, 1]
n_cells_values = [50, 100, 500, 5000]
ncells_results = {}

for nc in n_cells_values:
    print(f"  Simulating N_cells={nc}...")
    res_nc, _ = simulate_uniform_gene_specificity(
        Anno, exp_mat,
        n_sims=200,
        theta=global_theta,
        seed=42,
        n_cells_override=nc,
        max_cells_per_type=nc,
    )
    ncells_results[nc] = res_nc

colors_nc = plt.cm.viridis(np.linspace(0.2, 0.9, len(n_cells_values)))
for i, (nc, res_nc) in enumerate(ncells_results.items()):
    ax.scatter(
        res_nc["total_umi"], res_nc["std_spec"],
        color=colors_nc[i], alpha=0.3, s=15, edgecolors="none",
        label=f"N = {nc}", zorder=3 + i,
    )
    # Binned trend
    bins = np.logspace(np.log10(res_nc["total_umi"].min()),
                       np.log10(res_nc["total_umi"].max()), 15)
    res_nc_copy = res_nc.copy()
    res_nc_copy["umi_bin"] = pd.cut(res_nc_copy["total_umi"], bins=bins)
    binned = res_nc_copy.groupby("umi_bin", observed=True)["std_spec"].median()
    bin_centers = [(b.left + b.right) / 2 for b in binned.index]
    ax.plot(bin_centers, binned.values, color=colors_nc[i], lw=2, zorder=10 + i)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total UMI per cell type")
ax.set_ylabel("Specificity noise (SD)")
ax.set_title("B", fontweight="bold", loc="left", fontsize=16)
ax.legend(fontsize=9, framealpha=0.8, title="Cells per type", title_fontsize=9)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# =========================================================================
# Panel C: Effect of Specificity Caps
# =========================================================================
ax = axes[1, 0]
cap_levels = [1.0, 2.0, 3.0, None]
cap_labels = ["Cap = 1×", "Cap = 2×", "Cap = 3×", "Uncapped"]
cap_colors = ["#999999", "#2563eb", "#16a34a", "#dc2626"]

ct_ids = sorted(Anno.index)
total_umis_arr = np.array([Anno.loc[ct, "Total UMI"] for ct in ct_ids])

for cap_val, label, color in zip(cap_levels, cap_labels, cap_colors):
    if cap_val is not None:
        capped_specs = np.clip(all_specs, 0, cap_val)
    else:
        capped_specs = all_specs.copy()

    mean_capped = capped_specs.mean(axis=0)
    ax.scatter(total_umis_arr, mean_capped, color=color, alpha=0.3, s=15,
               edgecolors="none", label=label, zorder=3)

    df_tmp = pd.DataFrame({"umi": total_umis_arr, "spec": mean_capped})
    bins = np.logspace(np.log10(total_umis_arr.min()),
                       np.log10(total_umis_arr.max()), 15)
    df_tmp["bin"] = pd.cut(df_tmp["umi"], bins=bins)
    binned = df_tmp.groupby("bin", observed=True)["spec"].median()
    bin_centers = [(b.left + b.right) / 2 for b in binned.index]
    ax.plot(bin_centers, binned.values, color=color, lw=2, zorder=10)

ax.axhline(y=1.0, color="gray", ls="--", lw=1.5, alpha=0.7)
ax.set_xscale("log")
ax.set_xlabel("Total UMI per cell type")
ax.set_ylabel("Mean simulated specificity")
ax.set_title("C", fontweight="bold", loc="left", fontsize=16)
ax.legend(fontsize=9, framealpha=0.8, loc="upper right")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# =========================================================================
# Panel D: Validation — Simulated Inflation vs Empirical frac_clipped
# =========================================================================
ax = axes[1, 1]

spec_unclip = pd.read_csv(SPEC_UNCLIP_PATH, index_col=0)
spec_unclip.columns = [int(c) for c in spec_unclip.columns]
clip_threshold = np.mean(spec_unclip.values.flatten()) * 2

# Vectorized frac_clipped computation
empirical_frac_clipped = (spec_unclip > clip_threshold).mean(axis=0)

# Simulated noise: use SD of specificity across simulations as predictor.
# High SD = more genes randomly pushed to extreme values = more clipping.
sim_noise = sim_result.set_index("ct")["std_spec"]

validation_df = pd.DataFrame({
    "sim_noise": sim_noise,
    "frac_clipped": empirical_frac_clipped,
    "total_umi": [Anno.loc[ct, "Total UMI"] for ct in Anno.index],
    "is_neuronal": [ct in Neur_idx for ct in Anno.index],
}, index=Anno.index).dropna()

neur_val = validation_df["is_neuronal"]
ax.scatter(
    validation_df.loc[neur_val, "sim_noise"],
    validation_df.loc[neur_val, "frac_clipped"],
    color="red", alpha=0.4, s=25, edgecolors="white", lw=0.3,
    label="Neuronal", zorder=3,
)
ax.scatter(
    validation_df.loc[~neur_val, "sim_noise"],
    validation_df.loc[~neur_val, "frac_clipped"],
    color="blue", alpha=0.6, s=25, edgecolors="white", lw=0.3,
    label="Non-neuronal", zorder=4,
)

rho_val, p_val = stats.spearmanr(validation_df["sim_noise"], validation_df["frac_clipped"])
ax.text(0.03, 0.97, f"ρ = {rho_val:.3f}\np = {p_val:.1e}",
        transform=ax.transAxes, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax.set_xlabel("Simulated specificity noise (SD)")
ax.set_ylabel("Empirical fraction of genes\nexceeding cap")
ax.set_title("D", fontweight="bold", loc="left", fontsize=16)
ax.legend(fontsize=9, framealpha=0.8, loc="lower right")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout(h_pad=3, w_pad=3)
fig.savefig(FIG_DIR / "zinb_simulation_main.pdf",
            dpi=300, transparent=True, bbox_inches="tight")
fig.savefig(FIG_DIR / "zinb_simulation_main.png",
            dpi=300, transparent=True, bbox_inches="tight")
plt.show()
print(f"Saved main figure to {FIG_DIR}")

## Step 4: Quantitative Validation

Show that the simulation's predictions match the empirical observations:
- Cell types with highest simulated specificity inflation = highest empirical `frac_clipped`
- Strong correlation between simulated inflation and empirical `frac_clipped`

In [ ]:
print("=" * 60)
print("VALIDATION: Simulated Noise vs Empirical frac_clipped")
print("=" * 60)

rho, p = stats.spearmanr(validation_df["sim_noise"], validation_df["frac_clipped"])
print(f"\nSpearman(sim_noise, frac_clipped): ρ = {rho:.3f}, p = {p:.2e}")

rho_umi, p_umi = stats.spearmanr(validation_df["total_umi"], validation_df["sim_noise"])
print(f"Spearman(Total UMI, sim_noise): ρ = {rho_umi:.3f}, p = {p_umi:.2e}")

print("\nTop 10 cell types by simulated noise:")
top_sim = validation_df.nlargest(10, "sim_noise")
for ct, row in top_sim.iterrows():
    sc = Anno.loc[ct, "Supercluster"]
    neur = "NEUR" if row["is_neuronal"] else "non-NEUR"
    print(f"  CT {ct}: sim_noise={row['sim_noise']:.4f}, "
          f"frac_clipped={row['frac_clipped']:.3f}, UMI={row['total_umi']:.0f} "
          f"[{sc}] ({neur})")

print("\nTop 10 cell types by empirical frac_clipped:")
top_emp = validation_df.nlargest(10, "frac_clipped")
for ct, row in top_emp.iterrows():
    sc = Anno.loc[ct, "Supercluster"]
    neur = "NEUR" if row["is_neuronal"] else "non-NEUR"
    print(f"  CT {ct}: frac_clipped={row['frac_clipped']:.3f}, "
          f"sim_noise={row['sim_noise']:.4f}, UMI={row['total_umi']:.0f} "
          f"[{sc}] ({neur})")

top20_sim = set(validation_df.nlargest(20, "sim_noise").index)
top20_emp = set(validation_df.nlargest(20, "frac_clipped").index)
overlap = top20_sim & top20_emp
print(f"\nOverlap in top 20: {len(overlap)}/20 ({len(overlap)/20:.0%})")

In [ ]:
print("\nNeuronal vs Non-neuronal Specificity Noise:")
neur_noise = validation_df.loc[neur_val, "sim_noise"]
nonneur_noise = validation_df.loc[~neur_val, "sim_noise"]
U, p_mw = stats.mannwhitneyu(neur_noise, nonneur_noise, alternative="less")
print(f"  Neuronal mean noise (SD): {neur_noise.mean():.4f} ± {neur_noise.std():.4f}")
print(f"  Non-neuronal mean noise (SD): {nonneur_noise.mean():.4f} ± {nonneur_noise.std():.4f}")
print(f"  Mann-Whitney (neur < non-neur): U={U:.0f}, p={p_mw:.2e}")

## Summary

### Key Findings

1. **Sampling noise inflates specificity in low-UMI cell types**: For uniformly expressed
   genes, small non-neuronal clusters (B cells, T cells, microglia) show specificity up to
   ~1.5–2× the true value, while large neuronal clusters remain near 1.0.

2. **NB noise model fits real data**: The mean-variance relationship shows Var/Mean > 1
   across all cluster sizes, confirming NB (not Poisson) is the appropriate noise model.
   Fitted θ values provide empirically grounded simulation parameters.

3. **More cells reduce noise but can't eliminate UMI-depth bias**: Even with 5,000 cells
   per type, low-UMI clusters still show inflated specificity because the bias comes from
   the per-cell UMI depth, not just sample size.

4. **Cap at 2× effectively neutralizes inflation**: A specificity cap of 2× brings all
   cell types close to the true value of 1.0, while preserving real biological signal.
   Cap at 1× is too aggressive; cap at 3× leaves some residual inflation.

5. **Simulation predictions match empirical data**: Cell types predicted to have the
   highest specificity inflation are the same ones with the highest `frac_clipped` in
   real data, with strong Spearman correlation.